# LeetCode #1015: Swim in Rising Water

https://leetcode.com/problems/swim-in-rising-water/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^4)$ | $O(n^2)$ |
| **Binary Search + DFS** | $O(n^2 \log n)$ | $O(n^2)$ |
| **Optimal: Dijkstra (Min-Heap) ★** | $O(n^2 \log n)$ | $O(n^2)$ |

---

## Understanding the Methods

### Brute Force
Try every possible time value $t$ from 0 to $n^2-1$, and for each $t$ run a BFS/DFS to see if a path exists where all cells have elevation $\le t$. This is correct but extremely slow.

### Binary Search + DFS
Binary search on the answer $t$, and for each candidate $t$ run a DFS to check reachability. Same asymptotic complexity as Dijkstra but with a larger constant due to repeated DFS calls.

### Optimal: Dijkstra (Min-Heap) ★
Use a min-heap priority queue seeded with $(elevation[0][0], 0, 0)$. Greedily expand the lowest-elevation unvisited cell; track the running maximum elevation seen. The answer is this maximum when we reach $(n-1, n-1)$ — we reach the target at the earliest possible time with no wasted steps.

**Why this is better than Binary Search + DFS:** Dijkstra finds the answer in a single pass whereas binary search reruns DFS $O(\log n)$ times; both are $O(n^2 \log n)$ but Dijkstra has a lower constant.

**Constraints:**
* $1 \le n \le 50$
* $grid[i][j]$ is a permutation of $[0, n^2 - 1]$

## Solutions

### C#

In [ ]:
using System.Collections.Generic;
public class Solution {
    public int SwimInWater(int[][] grid) {
        int n = grid.Length;
        // Min-heap keyed by max elevation on path so far
        var pq = new PriorityQueue<(int elev, int r, int c), int>();
        var visited = new bool[n, n];
        pq.Enqueue((grid[0][0], 0, 0), grid[0][0]);
        int[] dr = { -1, 1, 0, 0 };
        int[] dc = { 0, 0, -1, 1 };
        while (pq.Count > 0) {
            var (elev, r, c) = pq.Dequeue();
            if (visited[r, c]) continue;
            visited[r, c] = true;
            // Arriving at destination means we have the minimum time
            if (r == n - 1 && c == n - 1) return elev;
            for (int d = 0; d < 4; d++) {
                int nr = r + dr[d], nc = c + dc[d];
                if (nr < 0 || nr >= n || nc < 0 || nc >= n || visited[nr, nc]) continue;
                // Cost is max elevation encountered along the path
                int next = Math.Max(elev, grid[nr][nc]);
                pq.Enqueue((next, nr, nc), next);
            }
        }
        return -1; // unreachable
    }
}

### Python

In [ ]:
import heapq
class Solution:
    def swim_in_water(self, grid: list[list[int]]) -> int:
        n = len(grid)
        # Min-heap keyed by max elevation on path so far
        heap = [(grid[0][0], 0, 0)]
        visited = [[False] * n for _ in range(n)]
        while heap:
            elev, r, c = heapq.heappop(heap)
            if visited[r][c]:
                continue
            visited[r][c] = True
            # Arriving at destination means we have the minimum time
            if r == n - 1 and c == n - 1:
                return elev
            for nr, nc in [(r-1,c),(r+1,c),(r,c-1),(r,c+1)]:
                if 0 <= nr < n and 0 <= nc < n and not visited[nr][nc]:
                    # Cost is max elevation encountered along the path
                    heapq.heappush(heap, (max(elev, grid[nr][nc]), nr, nc))
        return -1

### Go

In [ ]:
import "container/heap"
type Item struct { elev, r, c int }
type PQ []Item
func (h PQ) Len() int            { return len(h) }
func (h PQ) Less(i, j int) bool  { return h[i].elev < h[j].elev }
func (h PQ) Swap(i, j int)       { h[i], h[j] = h[j], h[i] }
func (h *PQ) Push(x interface{}) { *h = append(*h, x.(Item)) }
func (h *PQ) Pop() interface{}   { old := *h; n := len(old); x := old[n-1]; *h = old[:n-1]; return x }
func swimInWater(grid [][]int) int {
    n := len(grid)
    visited := make([][]bool, n)
    for i := range visited { visited[i] = make([]bool, n) }
    h := &PQ{{grid[0][0], 0, 0}}
    heap.Init(h)
    dr := []int{-1, 1, 0, 0}
    dc := []int{0, 0, -1, 1}
    for h.Len() > 0 {
        item := heap.Pop(h).(Item)
        elev, r, c := item.elev, item.r, item.c
        if visited[r][c] { continue }
        visited[r][c] = true
        // Arriving at destination means we have the minimum time
        if r == n-1 && c == n-1 { return elev }
        for d := 0; d < 4; d++ {
            nr, nc := r+dr[d], c+dc[d]
            if nr < 0 || nr >= n || nc < 0 || nc >= n || visited[nr][nc] { continue }
            // Cost is max elevation encountered along the path
            next := elev
            if grid[nr][nc] > next { next = grid[nr][nc] }
            heap.Push(h, Item{next, nr, nc})
        }
    }
    return -1
}

### Rust

In [ ]:
use std::collections::BinaryHeap;
use std::cmp::Reverse;
impl Solution {
    pub fn swim_in_water(grid: Vec<Vec<i32>>) -> i32 {
        let n = grid.len();
        let mut visited = vec![vec![false; n]; n];
        // Min-heap via Reverse; stores (max_elev, row, col)
        let mut heap = BinaryHeap::new();
        heap.push(Reverse((grid[0][0], 0usize, 0usize)));
        let dirs: &[(i32, i32)] = &[(-1,0),(1,0),(0,-1),(0,1)];
        while let Some(Reverse((elev, r, c))) = heap.pop() {
            if visited[r][c] { continue; }
            visited[r][c] = true;
            // Arriving at destination means we have the minimum time
            if r == n - 1 && c == n - 1 { return elev; }
            for &(dr, dc) in dirs {
                let nr = r as i32 + dr;
                let nc = c as i32 + dc;
                if nr < 0 || nr >= n as i32 || nc < 0 || nc >= n as i32 { continue; }
                let (nr, nc) = (nr as usize, nc as usize);
                if visited[nr][nc] { continue; }
                // Cost is max elevation encountered along the path
                heap.push(Reverse((elev.max(grid[nr][nc]), nr, nc)));
            }
        }
        -1
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `grid = [[0,2],[1,3]]`
Dijkstra starts at cell 0 (elevation 0). It can move to cell 1 (elevation 2) or cell 2 (elevation 1). The path $0 \to 2 \to 3$ gives $\max = 3$; the path $0 \to 1 \to 3$ also gives $\max = 3$. Answer: **3**.

### 2. Slightly Complex
**Input:** `grid = [[0,1,2],[3,8,4],[9,5,6]]`
Optimal path avoids elevation 8 by going right then down, yielding $\max(0,1,2,4,6) = 6$. Greedily picking the lowest-cost neighbor at each step, the heap naturally finds this route. Answer: **6**.

### 3. Edge Case: Time Factor
**Input:** `grid = [[0,N-1,...], ..., [..., N^2-1]]` (large $n=50$)
With $n=50$ there are $2500$ cells; the heap processes at most $O(n^2)$ items with $O(\log n^2)$ work each, so the bottleneck is the heap size $-$ total $O(n^2 \log n)$ operations.

### 4. Edge Case: Space Factor
**Input:** `grid` where every cell has a distinct large elevation forcing a zig-zag path.
The visited array and heap together consume $O(n^2)$ space; in the worst case all $n^2$ cells are enqueued before a single cell is finalized.

### 5. Almost-Impossible but Plausible
**Input:** `grid = [[0]]` ($n=1$)
Start equals destination. The heap pops the single element immediately; result is $grid[0][0] = 0$ with no neighbors processed. No iteration needed.